# Phase 3 – File 6: Taylor Series & Second-Order Approximations

**Understanding local polynomial approximations, Newton-Raphson optimization, and why Deep Learning relies on First-Order methods.**

---

## 1. What is a Taylor Series?

Any smooth, differentiable function $f(x)$ can be approximated near a point $x_0$ as an infinite polynomial using its derivatives evaluated at $x_0$:

$$f(x) = f(x_0) + f'(x_0)(x - x_0) + \frac{f''(x_0)}{2!}(x - x_0)^2 + \frac{f'''(x_0)}{3!}(x - x_0)^3 + \dots = \sum_{k=0}^\infty \frac{f^{(k)}(x_0)}{k!} (x - x_0)^k$$

- **0th Order**: $f(x) \approx f(x_0)$ (constant approximation)
- **1st Order**: $f(x) \approx f(x_0) + f'(x_0)(x - x_0)$ (linear tangent approximation)
- **2nd Order**: $f(x) \approx f(x_0) + f'(x_0)(x - x_0) + \frac{1}{2}f''(x_0)(x - x_0)^2$ (parabolic / quadratic approximation)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Visualizing Taylor Approximations of f(x) = cos(x) around x0 = 0
x = np.linspace(-4, 4, 300)
f_true = np.cos(x)

# Taylor polynomials:
# f(0) = 1, f'(0) = 0, f''(0) = -1, f'''(0) = 0, f''''(0) = 1
T0 = np.ones_like(x)
T1 = np.ones_like(x) # Same as T0 since f'(0)=0
T2 = 1 - (x**2) / 2
T4 = 1 - (x**2) / 2 + (x**4) / 24

plt.figure(figsize=(10, 6))
plt.plot(x, f_true, 'k-', linewidth=2.5, label=r'True $f(x) = \cos(x)$')
plt.plot(x, T0, 'r--', label=r'Order 0: $1$')
plt.plot(x, T2, 'b--', label=r'Order 2: $1 - \frac{x^2}{2}$')
plt.plot(x, T4, 'g-.', label=r'Order 4: $1 - \frac{x^2}{2} + \frac{x^4}{24}$')
plt.ylim(-2, 2.5)
plt.title(r"Taylor Series Approximations of $\cos(x)$ around $x_0 = 0$")
plt.xlabel("x"); plt.ylabel("y")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


---

## 2. Multivariable Taylor Expansion

For a multi-variable loss function $L(\mathbf{x}): \mathbb{R}^n \to \mathbb{R}$ around $\mathbf{x}_0$:

$$L(\mathbf{x}) \approx L(\mathbf{x}_0) + \nabla L(\mathbf{x}_0)^T (\mathbf{x} - \mathbf{x}_0) + \frac{1}{2} (\mathbf{x} - \mathbf{x}_0)^T \mathbf{H} (\mathbf{x} - \mathbf{x}_0)$$

where:
- $\nabla L(\mathbf{x}_0)$ is the **Gradient vector** (1st derivative).
- $\mathbf{H} = \nabla^2 L(\mathbf{x}_0)$ is the **Hessian matrix** (2nd derivative).


In [ ]:
# Quadratic approximation of 2D function
def f_2d(x1, x2):
    return np.sin(x1) * np.exp(0.5 * x2)

# Point of expansion x0 = [0, 0]
# f(0,0) = 0
# grad = [cos(0)*exp(0), sin(0)*0.5*exp(0)] = [1, 0]
# Hessian = [[-sin(0)*exp(0), cos(0)*0.5*exp(0)], [cos(0)*0.5*exp(0), sin(0)*0.25*exp(0)]] = [[0, 0.5], [0.5, 0]]

def quad_approx(x1, x2):
    return x1 + x1 * x2

x1 = np.linspace(-1, 1, 50)
x2 = np.linspace(-1, 1, 50)
X1, X2 = np.meshgrid(x1, x2)
Z_true = f_2d(X1, X2)
Z_approx = quad_approx(X1, X2)

fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.plot_surface(X1, X2, Z_true, cmap='viridis', alpha=0.8)
ax1.set_title("True Function f(x1, x2)")

ax2 = fig.add_subplot(1, 2, 2, projection='3d')
ax2.plot_surface(X1, X2, Z_approx, cmap='coolwarm', alpha=0.8)
ax2.set_title("2nd Order Taylor Approximation")
plt.tight_layout()
plt.show()


---

## 3. Newton-Raphson Optimization

### The Idea:
Instead of approximating a loss function with a flat tangent line (Gradient Descent), fit a local **parabola** (quadratic Taylor series) and jump straight to the vertex of the parabola!

Setting the gradient of the quadratic Taylor approximation to zero:
$$\nabla \left[ L(\mathbf{x}_0) + \nabla L(\mathbf{x}_0)^T (\mathbf{x} - \mathbf{x}_0) + \frac{1}{2} (\mathbf{x} - \mathbf{x}_0)^T \mathbf{H} (\mathbf{x} - \mathbf{x}_0) \right] = \mathbf{0}$$
$$\nabla L(\mathbf{x}_0) + \mathbf{H} (\mathbf{x} - \mathbf{x}_0) = \mathbf{0}$$

$$\mathbf{x}_{t+1} = \mathbf{x}_t - \mathbf{H}^{-1} \nabla L(\mathbf{x}_t)$$

### Comparison Table:

| Feature | First-Order (Gradient Descent) | Second-Order (Newton's Method) |
| :--- | :--- | :--- |
| **Update Rule** | $\mathbf{x}_{t+1} = \mathbf{x}_t - \alpha \nabla L(\mathbf{x}_t)$ | $\mathbf{x}_{t+1} = \mathbf{x}_t - \mathbf{H}^{-1} \nabla L(\mathbf{x}_t)$ |
| **Convergence Speed** | Linear $O(1/t)$ | **Quadratic $O(1/t^2)$** (rapid convergence) |
| **Learning Rate** | Required ($\alpha$) | **Self-scaling step size** |
| **Computation Cost** | $O(N)$ per step | **$O(N^3)$ per step** (inverting $N \times N$ Hessian) |
| **Memory Cost** | $O(N)$ | **$O(N^2)$** (storing $N \times N$ Hessian) |


In [ ]:
# Comparing Gradient Descent vs Newton's Method on a 1D function: f(x) = x^4 - 4x^2 + 4
def f(x): return x**4 - 4*x**2 + 4
def df(x): return 4*x**3 - 8*x
def d2f(x): return 12*x**2 - 8

# 1. Gradient Descent
x_gd = 2.5
lr = 0.02
gd_history = [x_gd]
for _ in range(15):
    x_gd = x_gd - lr * df(x_gd)
    gd_history.append(x_gd)

# 2. Newton's Method
x_newton = 2.5
newton_history = [x_newton]
for _ in range(5):
    x_newton = x_newton - df(x_newton) / d2f(x_newton)
    newton_history.append(x_newton)

print("GD after 15 steps:", np.round(gd_history[-1], 4))
print("Newton's Method after just 4 steps:", np.round(newton_history[-1], 4))


---

## 4. Why Deep Learning Uses 1st-Order Methods (SGD/Adam)

If Newton's method is so much faster in step count, why doesn't PyTorch use it for training LLMs or ResNets?

1. **Parameter Scale**: A modern LLM has $N = 7 \times 10^9$ (7 Billion) parameters.
   - Storing the Hessian $\mathbf{H}$ requires $N^2 = 4.9 \times 10^{19}$ floating point numbers $\approx 200,000$ Petabytes of GPU VRAM!
   - Inverting $\mathbf{H}$ takes $O(N^3) \approx 3.4 \times 10^{29}$ operations.
2. **Saddle Points**: If $\mathbf{H}$ has negative eigenvalues (common in non-convex neural landscapes), Newton's method actually jumps **towards local maxima or saddle points**!

Therefore, deep learning relies on **First-Order optimizers with Momentum and Adaptive Learning Rates (AdamW)**.


---

## 5. Summary & Key Takeaways

1. **Taylor Series** approximates non-linear functions locally using derivatives.
2. **1st-Order Taylor Series** leads to Gradient Descent.
3. **2nd-Order Taylor Series** incorporates the Hessian matrix and yields Newton's Method.
4. While Newton's method converges quadratically, its $O(N^3)$ computational cost and $O(N^2)$ memory footprint make first-order methods (SGD, Adam) the undisputed choice for modern deep learning.
